# Level 2 Extraction Experiments

This notebook evaluates field extraction quality (`vendor`, `date`, `total`) on the public dummy test set by running image-only inference.

In [ ]:
from pathlib import Path
import json
from solution import DocFusionSolution

repo = Path('/Users/abx/rihal/rihal-codestacker/ML')
submission = repo / 'my_submission'
sol = DocFusionSolution()
model_dir = sol.train(str(repo / 'dummy_data' / 'train'), str(submission / 'models' / 'nb_level2'))
model_dir

In [ ]:
test_rows = []
with (repo / 'dummy_data' / 'test' / 'test.jsonl').open() as f:
    for line in f:
        test_rows.append(json.loads(line))

preds = []
for row in test_rows:
    img = repo / 'dummy_data' / 'test' / row['image_path']
    pred = sol.analyze_image(model_dir, str(img))
    preds.append({'id': row['id'], 'truth': row['fields'], 'pred': pred})

preds[:2]

In [ ]:
def total_close(a, b, tol=0.01):
    try:
        return abs(float(a) - float(b)) <= tol
    except Exception:
        return False

vendor_acc = sum(int(p['truth']['vendor'] == p['pred'].get('vendor')) for p in preds) / len(preds)
date_acc = sum(int(p['truth']['date'] == p['pred'].get('date')) for p in preds) / len(preds)
total_acc = sum(int(total_close(p['truth']['total'], p['pred'].get('total'))) for p in preds) / len(preds)

{
    'vendor_exact_acc': vendor_acc,
    'date_exact_acc': date_acc,
    'total_close_acc': total_acc,
}